In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from dotenv import load_dotenv

from inference.evaluator import Evaluator
from inference.persona_registry import PersonaRegistry
from evaluate import extract_answers, test_vs_baseline

load_dotenv()

True

In [2]:
persona_registry = PersonaRegistry()
answer_columns = persona_registry.get_answer_columns()      # columns with completions by LLM
extracted_columns = [col.replace("_answer", "") for col in answer_columns]      # columns with extracted answers

base_col = "base"
persona_cols = [col for col in extracted_columns if col not in ["no", "helpful", "base"]]

In [5]:
data_folder = Path("data")
for dataset_path in data_folder.iterdir():
    path_name = dataset_path.name
    if "backup" in path_name:
        continue

    print(f"Running test: {path_name}")

    # Load data
    evaluator = Evaluator(dataset_path=path_name)
    mmlu_tasks, mmlu_df = evaluator.get_data("mmlu-pro")
    mmlu_df = extract_answers(mmlu_df, answer_columns, "mmlu")

    mmlu_effects_df = mmlu_df.copy()
    for col in extracted_columns:
        eq_col = mmlu_effects_df["answer"] == mmlu_effects_df[col]
        mmlu_effects_df[col] = eq_col.astype("int")

    results = test_vs_baseline(mmlu_effects_df, base_col, persona_cols)
    for r in results:
        print(f"- Persona {r['persona']:<20} stat {r['statistic']:>10.3f} p-value {r['p_value']:>10.4f}")

Running test: llama-3_2-1b-instruct
- Persona static_short         stat     85.500 p-value     1.0000
- Persona static_medium        stat     60.000 p-value     0.1083
- Persona static_long          stat    108.000 p-value     0.0499
- Persona dynamic_short        stat     78.000 p-value     0.0093
- Persona dynamic_medium       stat    153.000 p-value     0.0090
- Persona dynamic_long         stat    208.000 p-value     0.3692
- Persona beginner_teacher     stat    153.000 p-value     0.0090
- Persona intermediate_teacher stat    378.000 p-value     0.4349
- Persona expert_teacher       stat    132.000 p-value     0.0047
Running test: qwen3-4b
- Persona static_short         stat   2164.500 p-value     0.0023
- Persona static_medium        stat   3799.000 p-value     0.2195
- Persona static_long          stat   3168.000 p-value     0.0022
- Persona dynamic_short        stat   4239.500 p-value     0.1732
- Persona dynamic_medium       stat   5964.000 p-value     0.0530
- Persona dynamic